# 02 · Feature Engineering & Train/Val/Test Split

| Step | Description |
|---|---|
| **Feature pipeline** | Returns · Volatility · Volume · Market Structure (PySpark window functions) |
| **Target label** | Forward log-return via `F.lead()` — strictly no look-ahead |
| **Split** | Chronological 70/15/15 — absolutely no random shuffling |

**Reads:** `processed/btc_cleaned.parquet` · `processed/audit_meta.yaml`  
**Saves:** `processed/btc_features.parquet` · `processed/train.parquet` · `processed/val.parquet` · `processed/test.parquet` · `processed/split_meta.yaml`

In [ ]:
## Setup

import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ.pop("SPARK_HOME", None)   # Let pyspark 4.2.0 use its own bundled JARs

import sys, warnings, yaml
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_session   import get_spark
from src.features.pipeline    import build_feature_pipeline
from src.models.splitter      import chronological_split
from src.visualisation.eda_plots import plot_correlation_heatmap

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED  = PROJECT_ROOT / CFG['artifacts']['processed_dir']
FIGURES    = PROJECT_ROOT / CFG['reports']['figures_dir']
for d in [PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)
with open(PROCESSED / 'audit_meta.yaml') as f:
    META = yaml.safe_load(f)

TS_COL     = META['ts_col']
TARGET_COL = f'target_log_return_{CFG["features"]["target_horizon"]}'
PASSTHROUGH = {'open','high','low','close','volume', TS_COL}

print(f'Timestamp column : {TS_COL}')
print(f'Target column    : {TARGET_COL}')

---
## 1 · Feature Engineering

> ⚠️ **Anti-leakage**: all window functions use `rowsBetween(-(w-1), 0)` (past-only). Target label built with `F.lead()`.

In [ ]:
# ── Load cleaned data → Spark ─────────────────────────────────────────
df_clean = pd.read_parquet(PROCESSED / 'btc_cleaned.parquet')
df_clean[TS_COL] = pd.to_datetime(df_clean[TS_COL])

spark = get_spark()
sdf   = spark.createDataFrame(df_clean)
print(f'Loaded into Spark: {sdf.count():,} rows × {len(sdf.columns)} cols')

In [ ]:
# ── Run full feature pipeline ─────────────────────────────────────────
print('Building features (PySpark window functions)...')
sdf_features = build_feature_pipeline(sdf, ts_col=TS_COL, close_col='close')

# Persist to avoid recomputing the full DAG on both .count() and .toPandas()
sdf_features.persist()
print(f'Feature DataFrame: {sdf_features.count():,} rows × {len(sdf_features.columns)} columns')

In [ ]:
# ── Collect to pandas ─────────────────────────────────────────────────
# Disable Arrow for collection — avoids socket crashes on problematic batches.
# Row-by-row serialisation is slower but tolerates edge-case float values.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
df_features = sdf_features.toPandas()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

sdf_features.unpersist()   # release cached memory

df_features[TS_COL] = pd.to_datetime(df_features[TS_COL])
df_features = df_features.sort_values(TS_COL).reset_index(drop=True)

spark.stop()
print('Spark stopped.')

feature_cols = [c for c in df_features.columns
                if c not in PASSTHROUGH and not c.startswith('target_')]

print(f'\nFeature groups:')
for label, kw in [('Returns', 'return'), ('Volatility', 'vol'),
                  ('Volume', 'volume'), ('Market Structure', 'drawdown momentum slope range')]:
    cols = [c for c in feature_cols if any(k in c for k in kw.split())]
    print(f'  {label:<18}: {len(cols)} cols')
print(f'  Total features  : {len(feature_cols)}')

In [ ]:
# ── Feature × target correlation heatmap (top 20) ────────────────────
top_cols = feature_cols[:20] + [TARGET_COL]
fig = plot_correlation_heatmap(df_features, columns=top_cols,
                               title='Feature × Target Correlation (top 20 features)')
fig.write_html(str(FIGURES / 'feature_correlation.html'))
fig.show()

In [ ]:
out = PROCESSED / 'btc_features.parquet'
df_features.to_parquet(out, index=False)
print(f'Saved → {out}  ({df_features.shape})')

---
## 2 · Chronological Train / Val / Test Split

**Rule:** Strictly chronological — no random shuffling, ever.

In [ ]:
# ── Split ─────────────────────────────────────────────────────────────
train_df, val_df, test_df = chronological_split(df_features, ts_col=TS_COL)

print('\nSplit summary:')
for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'  {name:5}: {part[TS_COL].min().date()} → {part[TS_COL].max().date()}  '
          f'({len(part):,} rows)')

In [ ]:
# ── Prepare X / y arrays (drop NaN rows) ─────────────────────────────
all_model_cols = feature_cols + [TARGET_COL, TS_COL]
train_clean = train_df[all_model_cols].dropna()
val_clean   = val_df[all_model_cols].dropna()
test_clean  = test_df[all_model_cols].dropna()

X_train, y_train = train_clean[feature_cols], train_clean[TARGET_COL]
X_val,   y_val   = val_clean[feature_cols],   val_clean[TARGET_COL]
X_test,  y_test  = test_clean[feature_cols],  test_clean[TARGET_COL]

print(f'X_train: {X_train.shape}  |  X_val: {X_val.shape}  |  X_test: {X_test.shape}')

In [ ]:
# ── Target distribution per partition ────────────────────────────────
fig = go.Figure()
for name, y, col in [('Train', y_train,'#4FC3F7'),('Val',y_val,'#F7931A'),('Test',y_test,'#66BB6A')]:
    fig.add_trace(go.Histogram(x=y, name=name, histnorm='probability density',
                               opacity=0.6, marker_color=col))
fig.update_layout(barmode='overlay', title=f'Target Distribution — {TARGET_COL}',
                  template='plotly_dark', xaxis_title='Log Return')
fig.write_html(str(FIGURES / 'target_distribution.html'))
fig.show()

In [ ]:
# ── Save partitions + metadata ────────────────────────────────────────
train_clean.to_parquet(PROCESSED / 'train.parquet', index=False)
val_clean.to_parquet(  PROCESSED / 'val.parquet',   index=False)
test_clean.to_parquet( PROCESSED / 'test.parquet',  index=False)

split_meta = {
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'ts_col'       : TS_COL,
    'train_end'    : str(train_clean[TS_COL].max().date()),
    'val_end'      : str(val_clean[TS_COL].max().date()),
    'test_end'     : str(test_clean[TS_COL].max().date()),
}
with open(PROCESSED / 'split_meta.yaml', 'w') as f:
    yaml.dump(split_meta, f, default_flow_style=False)

print('Partitions saved.')
print('Next → 03_modelling.ipynb')